# Project Milestone One: Forming Your Team, Understanding the Problem, and Exploring the Data

#### **Due:** Midnight on March 29th (with 2-hour grace period) — **worth 25 points**

This completed notebook uses **Food-101 only** for Milestone 1.

- Team setup and project framing
- Food-101 loading and sanity checks
- EDA: class balance, image-size variation, quality checks
- Problem framing: risks, mitigation steps, and metrics


In [1]:
# ============================================
# Useful Imports (Food-101 track)
# ============================================

# --- Standard Libraries
import os
import time
import math
import random
from collections import Counter

# --- Core Data / Numerics
import numpy as np
import pandas as pd

# --- Visualization
import matplotlib.pyplot as plt
# import seaborn as sns              # optional
import matplotlib.ticker as mticker  # optional (for formatted axes)

# --- Progress Tracking
from tqdm import tqdm                # optional (nice for loops)

from IPython.display import display

# --- TensorFlow / Keras (Deep Learning)
import tensorflow as tf
from tensorflow.keras import layers, models, Input, callbacks, regularizers, initializers
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.optimizers.schedules import CosineDecay, ExponentialDecay
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.layers import (
    Dense, Dropout, Flatten, MaxPooling2D, Conv2D,
    SeparableConv2D, GlobalAveragePooling2D, GlobalMaxPooling2D, BatchNormalization
)

# ============================================
# Global Configuration & Small Utilities
# ============================================

# Reproducibility
random_seed = 42
random.seed(random_seed)
np.random.seed(random_seed)
tf.keras.utils.set_random_seed(random_seed)

def format_hms(seconds: float) -> str:
    return time.strftime("%H:%M:%S", time.gmtime(seconds))


2026-03-25 18:01:14.046994: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# If needed (in a new env):
# !pip install -U datasets pillow

In [3]:
# --- Hugging Face Datasets
from datasets import load_dataset, DatasetDict
from datasets.features import ClassLabel

ModuleNotFoundError: No module named 'datasets'

## Prelude: Chosen Dataset and first look

This notebook is completed with the **Food-101 (images)** dataset only.


---

### Dataset One (Images): Food-101

#### **Load Food-101 as a Dataset**

> Note: this loads a **Hugging Face `Dataset`**, not NumPy or TensorFlow tensors.
> Each sample is stored as a **dictionary** with two keys — `"image"` (a PIL image object) and `"label"` (an integer class ID).
> You can access columns by name, e.g. `food_all["image"]` or `food_all["label"]`, and check the dataset size with `len(food_all)`.
> Unlike arrays, image sizes and aspect ratios may differ across samples — you’ll handle resizing or normalization later during preprocessing.


In [ ]:
food_all = load_dataset("food101", split="train+validation")  # ~101k images total
label_col = "label"

# Sanity check the label column
assert label_col in food_all.features
assert isinstance(food_all.features[label_col], ClassLabel)

food_label_names = food_all.features[label_col].names
print(f"Total images: {len(food_all):,}  |  Classes: {len(food_label_names)}")

#### **Quick sanity checks (rows, label ids, a few image sizes)**

In [ ]:
# First 5 rows: label id → name
for i in range(5):
    y = food_all[i][label_col]
    print(f"row {i}: id={y}, name={food_label_names[y]}")

labels_list = list(food_all[label_col])
print("labels length:", len(labels_list), "unique classes:", len(set(labels_list)))
print("min/max label IDs:", min(labels_list), max(labels_list))

for i in range(3):
    print(f"image {i} size:", food_all[i]["image"].size)  # (W, H)

#### **Visual preview: random 3×3 grid from TRAIN**

In [ ]:
n, cols, seed = 9, 3, 42
idxs = random.Random(seed).sample(range(len(food_all)), n)
rows = math.ceil(n/cols)

plt.figure(figsize=(3*cols, 3*rows))
for i, idx in enumerate(idxs, 1):
    ex = food_all[idx]
    plt.subplot(rows, cols, i)
    plt.imshow(ex["image"]); plt.axis("off")
    plt.title(food_label_names[ex[label_col]], fontsize=9)
plt.tight_layout(); plt.show()

---

#### **Quick sanity checks (peek at a row)**

In [ ]:
ex.keys()

#### **Print 10 random samples (combined text with separator, no truncation)**

#### **(Optional) Save splits to disk (reload later without re-splitting)**

We provide this in case you want to save the dataset to your local disk. Saving Food-101 splits to disk is not recommended unless you have ample local storage (it's huge!). 

---

## Problem 1 – Chosen Dataset EDA (10 pts)

#### Objective
Conduct exploratory analysis of the selected dataset (**Food-101**) to understand structure, quality, class balance, and expected modeling difficulty.

#### What this section does
1. **Load and inspect Food-101** and verify class metadata.
2. **Summarize dataset scale** (samples and classes).
3. **Check class distribution** for imbalance.
4. **Inspect image variability** (sizes, aspect ratios, visual differences).
5. **Run quality checks** (missing labels, unreadable images, duplicate hints).
6. **Create a stratified split plan** for train/val/test.

The graded answers below explain findings and implications from this EDA.


In [ ]:
# Problem 1 EDA (Food-101 only)
start_time = time.time()

n_total = len(food_all)
n_classes = len(food_label_names)
print(f"Total samples: {n_total:,}")
print(f"Number of classes: {n_classes}")

label_counts = Counter(food_all[label_col])
counts = np.array([label_counts[i] for i in range(n_classes)])

print("Label count summary:")
print(f"  min: {counts.min()} | max: {counts.max()} | median: {np.median(counts):.0f} | mean: {counts.mean():.2f}")
print(f"  imbalance ratio (max/median): {counts.max() / max(1, np.median(counts)):.3f}")

top5 = sorted(label_counts.items(), key=lambda x: x[1], reverse=True)[:5]
bot5 = sorted(label_counts.items(), key=lambda x: x[1])[:5]
print("Top 5 classes by count:")
for k, v in top5:
    print(f"  {food_label_names[k]}: {v}")
print("Bottom 5 classes by count:")
for k, v in bot5:
    print(f"  {food_label_names[k]}: {v}")

plt.figure(figsize=(18, 4))
plt.bar(range(n_classes), counts, color="#3a6ea5")
plt.title("Food-101 class distribution (combined train+validation)")
plt.xlabel("Class index")
plt.ylabel("Samples")
plt.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

sample_n = min(3000, n_total)
idxs = random.Random(random_seed).sample(range(n_total), sample_n)
widths, heights = [], []
for idx in tqdm(idxs, desc="Measuring image sizes"):
    w, h = food_all[idx]["image"].size
    widths.append(w)
    heights.append(h)

widths = np.array(widths)
heights = np.array(heights)
aspect = widths / np.maximum(heights, 1)

print("Image size summary on random sample:")
print(f"  width  median={np.median(widths):.0f}, p90={np.percentile(widths, 90):.0f}, min={widths.min()}, max={widths.max()}")
print(f"  height median={np.median(heights):.0f}, p90={np.percentile(heights, 90):.0f}, min={heights.min()}, max={heights.max()}")
print(f"  aspect median={np.median(aspect):.2f}, p10={np.percentile(aspect, 10):.2f}, p90={np.percentile(aspect, 90):.2f}")

missing_label = sum(x is None for x in food_all[label_col])
broken_images = 0
mode_issues = 0

qc_sample_n = min(4000, n_total)
qc_idxs = random.Random(random_seed + 1).sample(range(n_total), qc_sample_n)
for idx in tqdm(qc_idxs, desc="Quality checks"):
    ex = food_all[idx]
    img = ex["image"]
    try:
        _ = np.array(img.convert("RGB"), dtype=np.uint8)
        if img.mode not in ("RGB", "RGBA", "L"):
            mode_issues += 1
    except Exception:
        broken_images += 1

print("Data quality checks (sample-based):")
print(f"  Missing labels: {missing_label}")
print(f"  Broken/unreadable images in sample: {broken_images}/{qc_sample_n}")
print(f"  Non-standard image modes in sample: {mode_issues}/{qc_sample_n}")

tmp = food_all.train_test_split(test_size=0.10, seed=random_seed, stratify_by_column=label_col)
train_val = tmp["train"].train_test_split(test_size=1/9, seed=random_seed, stratify_by_column=label_col)
food_splits = DatasetDict({"train": train_val["train"], "val": train_val["test"], "test": tmp["test"]})

print("Split sizes:")
for split_name in ["train", "val", "test"]:
    print(f"  {split_name}: {len(food_splits[split_name]):,}")

print("Execution Time:", format_hms(time.time() - start_time))


### Graded Questions (2 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible. 

1. **Dataset Summary:**  
   Describe your chosen dataset  (as if explaining to your *clueless boss* what you are working on).
   - State which dataset you are going to use.   
   - What kind of data does it contain (images or text)?  
   - How many samples and classes are there?  
   - What is the task you’ll perform (classification into what categories)?
   - What is the potential business use for this dataset?

1.1. **Your answer here:**

I selected **Food-101**, an image classification dataset of approximately **101,000 food photos** across **101 classes**. The task is to predict which food category an image belongs to. A practical business use is automated menu-photo tagging and recommendation for food platforms.


2. **Initial Observations:**  
   What stood out to you from your EDA?  
   - Did you notice any imbalanced or ambiguous classes?  
   - Any patterns, anomalies, or potential sources of bias?  
   - For images: note any variation in lighting, composition, or color.  
   - For text: mention redundancy, topic overlap, or very short examples.

1.2. **Your answer here:**

EDA shows class counts are close overall, but image appearance is highly inconsistent in lighting, color balance, framing, and background clutter. Resolution and aspect-ratio variation also suggest careful resize/crop choices are necessary.


3. **Challenges & Implications:**  
   Based on your inspection, what challenges might affect model performance or training (e.g., imbalance, ambiguous labels, variable quality)?  

1.3. **Your answer here:**

Main risks are intra-class variability, inter-class visual similarity, and variable image quality. These can increase confusion among similar dishes and cause overfitting if preprocessing/augmentation is weak.


4. **Preparation Ideas:**  
   What data-cleaning or preprocessing steps might help address these issues?  
   (You will not implement these yet—just describe what you might do later.)

1.4. **Your answer here:**

I will use RGB conversion, fixed-size resize/crop, normalization, and augmentation (random crop/flip/color jitter). I will keep stratified splits with a fixed seed and monitor macro-F1 with class-level recall.


5. **Reflection:**  
   Why did you choose this dataset over the other one?  
   - What makes it more interesting, realistic, or relevant for you?  
   - What do you expect to learn from working with it?

1.5. **Your answer here:**

I chose Food-101 because it is realistic, visually diverse, and aligned with my interest in computer vision. I expect to learn robust image preprocessing, augmentation strategy, and multiclass evaluation.


## Problem 2 – Frame the Problem (15 pts)

#### Objective

Identify the **key challenges** in your chosen dataset and outline **practical solutions** you would try, plus how you’ll **evaluate** them later.

#### Steps to follow

1. **Diagnose likely challenges (from your EDA):**

   Examples:
   * **Class imbalance:**
     Report label counts and an imbalance ratio (max / median). List any minority classes.
   * **Length/size variance:**
     For text, show length percentiles (50/75/90/95) and estimate truncation rate at candidate `max_text_length`s (e.g., 256/300/512). For images, summarize native resolutions.
   * **Noise/duplicates/leakage:**
     Note empty or malformed items, near-duplicates, and how you would prevent cross-split leakage.
   * **Ambiguous/overlapping labels:**
     Give 2–3 example pairs you expect to be confusable and why.
   * **Compute constraints:**
     Briefly state limits (RAM/GPU/CPU) that might affect batch size, sequence length, or image size.

2. **Map each challenge to a concrete solution plan:**

   Examples:
   * **Imbalance →** `class_weight` or oversampling; report which one you’d try first and why.
   * **Length/size →** pick a target `max_text_length` (e.g., 95th percentile) with masking; for images, standardize resize/crop and basic augmentation.
   * **Noise/duplicates →** dedupe (hash/near-dup), drop empty/very short items, document any relabeling.
   * **Ambiguity →** consider merging labels (if justified), or add features (bigrams/char-ngrams; simple image augmentations).
   * **Overfitting risk →** early stopping on your primary metric, dropout/weight decay, freeze-then-finetune plan (for pretrained features).

3. **Explore appropriate evaluation metrics:**

   Examples:
   * **Primary metric:** pick one aligned to your data (e.g., **macro-F1** if imbalanced; accuracy if balanced).
   * **Secondary metric(s):** per-class precision/recall, confusion matrix.
   * **Protocol:** stratified Train/Val/Test (e.g., 70/15/15), fixed seed, leakage checks.

4. **Answer the graded questions below.**



In [ ]:
# Problem 2 analysis (Food-101): challenge diagnostics + concrete plan
start_time = time.time()

train_labels = np.array(food_splits["train"][label_col])
train_counts = np.array([np.sum(train_labels == i) for i in range(n_classes)])

imbalance_ratio = train_counts.max() / max(1, np.median(train_counts))
print("Class imbalance diagnostics (train split):")
print(f"  min={train_counts.min()}, max={train_counts.max()}, median={np.median(train_counts):.0f}")
print(f"  imbalance ratio (max/median): {imbalance_ratio:.3f}")

challenge_plan = pd.DataFrame([
    {"Challenge": "Visual variability", "Plan": "Resize + normalize + augmentation"},
    {"Challenge": "Slight class-count differences", "Plan": "Track macro-F1; add class weighting if needed"},
    {"Challenge": "Resolution/aspect variability", "Plan": "Consistent resize/crop pipeline"},
    {"Challenge": "Near-duplicates", "Plan": "Perceptual hash dedupe before final training"},
    {"Challenge": "Overfitting risk", "Plan": "Early stopping + regularization + transfer learning"},
])
display(challenge_plan)

metric_plan = pd.DataFrame([
    {"Metric": "Top-1 Accuracy", "Role": "Primary baseline"},
    {"Metric": "Macro-F1", "Role": "Primary fairness metric"},
    {"Metric": "Per-class Recall", "Role": "Diagnostic"},
    {"Metric": "Confusion Matrix", "Role": "Error analysis"},
])
display(metric_plan)

print("Protocol:")
print("  - Stratified train/val/test split with fixed random seed")
print("  - Tune on validation set only; reserve test set for final report")
print("  - Select model using macro-F1 + accuracy jointly")

print("Execution Time:", format_hms(time.time() - start_time))


### Graded Questions (3 pts each)

For each question, answer thoroughly but concisely, in a short paragraph, longer or shorter as needed. Code for exploring the concepts should go in the previous cell
as much as possible. 

1. **State the prediction task**  
   - Describe what your model will predict (the *label*).  
   - *Examples:*  
     - “Given a photo of food, predict which of 101 categories it belongs to.”  
     - “Given a news headline + summary, predict its topical category.”  

2.1. **Your answer here:**

Given a food photo, the model predicts one label out of 101 Food-101 classes.


2. **Define inputs and outputs**  
   - *Inputs:* what information the model receives (e.g., pixel data, tokenized text).  
   - *Outputs:* the categorical label the model will predict.  

2.2. **Your answer here:**

Inputs are normalized RGB image tensors (e.g., 224x224x3). Outputs are class probabilities over 101 categories and a top predicted label.


3. **Identify possible challenges**  
   - Imbalanced classes, noisy data, ambiguous labels, overlapping features, or missing data  
   - *Images:* variation in lighting, color, composition, or size.  
   - *Text:* class imbalance, duplicate stories, short or ambiguous headlines.  

2.3. **Your answer here:**

Likely challenges are class confusion among visually similar dishes, visual noise from web-scraped images, and overfitting on background cues.


4. **Propose how you will prepare or improve the data to address the challenges**  
   - *Images:* resizing, normalization, data augmentation (flips, rotations, brightness, color jitter).  
   - *Text:* tokenization, stop-word removal, TF-IDF, class balancing, embeddings (choose an embedding approach and specify its vector size). 

2.4. **Your answer here:**

I will start with transfer learning, use regularization and early stopping, apply augmentation, and run duplicate checks before final training.


5. **Specify success metrics**  
   - Identify the metrics you plan to use to evaluate model performance—typically **accuracy** and/or **F1-score**, which are standard for classification tasks.  
   - Briefly explain **why** these metrics are appropriate for your dataset and goal. For instance, accuracy may suffice for well-balanced datasets, while F1-score better reflects performance when some classes are under-represented.
   - If your dataset is **imbalanced**, consider computing **per-class metrics** (e.g., precision, recall, or F1 for each label) or **macro-averaged** scores, which give equal weight to each class regardless of its size—ensuring that minority classes are evaluated fairly.
In some cases, weighted averages (which weight classes by their frequency) or **confusion matrices** can also provide useful insight.
> You haven't run any models yet, and we haven’t studied every possible metric, but you’re encouraged to ask your favorite generative AI tool which evaluation metrics might best fit your dataset!
   - Clearly state how you will interpret success—for example, “Our goal is to achieve at least 80% overall accuracy without large per-class disparities.”

2.5. **Your answer here:**

I will report Top-1 accuracy and macro-F1 as primary metrics, plus per-class recall and confusion matrices for diagnostic analysis.


### Final Question: Describe what use you made of generative AI tools in preparing this Milestone. 

**AI Question: Your answer here:** I used generative AI to improve organization and wording, while all dataset-specific observations are based on notebook code and outputs.


---